# Chapter 18: Tries

The programs from Chapter 18 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## The Idea

`trie.py`

In [1]:
"""A trie (prefix tree) for words. Each node maps a letter to a child;
a path from the root spells a prefix, and a flag marks whole words."""


class TrieNode:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children = {}
        self.is_word = False


class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        self._n = 0
        for w in words:
            self.insert(w)

    def __len__(self):
        return self._n

    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        if not node.is_word:
            node.is_word = True
            self._n += 1

    def _walk(self, prefix):
        """The node reached by spelling prefix, or None."""
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def __contains__(self, word):
        node = self._walk(word)
        return node is not None and node.is_word

    def has_prefix(self, prefix):
        return self._walk(prefix) is not None

    def complete(self, prefix, limit=None):
        """Words starting with prefix, in alphabetical order."""
        start = self._walk(prefix)
        out = []
        stack = [(start, prefix)] if start else []
        while stack and (limit is None or len(out) < limit):
            node, word = stack.pop()
            if node.is_word:
                out.append(word)
            for ch in sorted(node.children, reverse=True):
                stack.append((node.children[ch], word + ch))
        return out

## Autocomplete

`autocomplete.py`

In [2]:
"""Autocomplete: suggest words as the user types."""

from trie import Trie

words = """apple apply apricot banana band bandana bank banner
           can candle candy cane cannon canvas""".split()
t = Trie(words)
for typed in ["ap", "ban", "can", "cand", "cat"]:
    print(f"{typed!r:>7}: {t.complete(typed, limit=3)}")

   'ap': ['apple', 'apply', 'apricot']
  'ban': ['banana', 'band', 'bandana']
  'can': ['can', 'candle', 'candy']
 'cand': ['candle', 'candy']
  'cat': []


## Timing

`bench_trie.py`

In [3]:
"""Is there any word starting with a prefix, among n words? A scan
with startswith(), our Trie, and binary search in a sorted list."""

import bisect
import random

import bench
from trie import Trie

rng = random.Random(2026)


def words(n):
    return sorted(
        {"".join(rng.choices("abcdefghij", k=8)) for _ in range(n)}
    )


def scan(ws, prefix):
    return any(w.startswith(prefix) for w in ws)


def sorted_search(ws, prefix):
    # the first word >= prefix is the only one that can start with it
    i = bisect.bisect_left(ws, prefix)
    return i < len(ws) and ws[i].startswith(prefix)


PREFIX = "jjjj"  # sorts last: a scan has to look at every word
sizes = [10_000, 20_000, 40_000, 80_000]
results = {
    "scan": bench.doubling(scan, lambda n: (words(n), PREFIX), sizes),
    "Trie": bench.doubling(
        Trie.has_prefix, lambda n: (Trie(words(n)), PREFIX), sizes
    ),
    "sorted + bisect": bench.doubling(
        sorted_search, lambda n: (words(n), PREFIX), sizes
    ),
}
bench.table(results)

         n            scan            Trie  sorted + bisect
    10,000          292 us          210 ns           221 ns
    20,000     660 us x2.3     212 ns x1.0      206 ns x0.9
    40,000    1.37 ms x2.1     210 ns x1.0      242 ns x1.2
    80,000    2.97 ms x2.2     209 ns x1.0      255 ns x1.1
